In [4]:
# Synthetic dataset generator for DeFi strategy-simulator (economic PvP via market)
# Creates multiple CSVs into OUT_DIR with progress prints.
# Requires: pandas, numpy

import os
import json
import time
import numpy as np
import pandas as pd

# ----------------------------
# CONFIG (tune these if you want faster runs)
# ----------------------------
SEED = 42
OUT_DIR = "synthetic_defi_game_data"

N_USERS = 20000     # try 3000 first to test speed
DAYS = 90           # try 30 first
START_DATE = "2025-01-01"

# ----------------------------
# CONSTANTS
# ----------------------------
RESOURCES = ["ORE", "ENERGY", "FOOD", "WATER", "CRYSTAL"]
COUNTRIES = ["US", "DE", "TR", "BR", "GB", "PL", "UA", "GE", "IN", "VN"]
CHANNELS = ["ads", "influencer", "organic", "referral", "partner"]
PLAYER_SEGMENTS = ["builder", "trader", "investor", "social"]
RISK = ["conservative", "balanced", "aggressive"]
WALLET_TYPES = ["EOA", "smart_wallet", "custodial"]
CHAINS = ["ETH", "ARB", "OP", "BSC", "POLY", "SOL"]

# ----------------------------
# UTIL: progress + tx_hash
# ----------------------------
t0 = time.time()
def prog(msg: str):
    print(f"[{time.time()-t0:7.1f}s] {msg}")

def prog_every(i: int, n: int, step: int, label: str):
    if i % step == 0 or i == n:
        pct = 100 * i / n if n else 100.0
        prog(f"{label}: {i}/{n} ({pct:.1f}%)")

def make_tx_hash(rng, nbytes: int = 32) -> str:
    # 32 bytes => 64 hex chars (EVM-like)
    return "0x" + rng.bytes(nbytes).hex()

# ----------------------------
# SETUP
# ----------------------------
rng = np.random.default_rng(SEED)

def dt(s):
    return pd.to_datetime(s)

start_dt = dt(START_DATE)
days = pd.date_range(start_dt, periods=DAYS, freq="D")
end_dt = days[-1] + pd.Timedelta(days=1)

os.makedirs(OUT_DIR, exist_ok=True)
prog(f"Start. OUT_DIR={OUT_DIR}, N_USERS={N_USERS}, DAYS={DAYS}")

# ----------------------------
# USERS
# ----------------------------
prog("Generating users...")

user_ids = np.arange(1, N_USERS + 1)

created_offsets = rng.integers(0, DAYS, size=N_USERS)
created_at = start_dt + pd.to_timedelta(created_offsets, unit="D") + pd.to_timedelta(rng.integers(0, 86400, N_USERS), unit="s")

acq_channel = rng.choice(CHANNELS, size=N_USERS, p=[0.35, 0.10, 0.35, 0.15, 0.05])
country = rng.choice(COUNTRIES, size=N_USERS)
timezone = rng.choice(["UTC", "UTC+1", "UTC+3", "UTC+4", "UTC-5", "UTC+7"], size=N_USERS)

device_os = rng.choice(["iOS", "Android", "Windows", "macOS", "Linux"], size=N_USERS, p=[0.25, 0.40, 0.20, 0.10, 0.05])
client = rng.choice(["mobile", "web", "desktop"], size=N_USERS, p=[0.55, 0.35, 0.10])
app_version = rng.choice(["1.0.0", "1.1.0", "1.2.0", "1.3.0"], size=N_USERS, p=[0.15, 0.25, 0.35, 0.25])

wallet_type = rng.choice(WALLET_TYPES, size=N_USERS, p=[0.75, 0.15, 0.10])
chain_pref = rng.choice(CHAINS, size=N_USERS, p=[0.35, 0.15, 0.10, 0.15, 0.15, 0.10])

player_segment = rng.choice(PLAYER_SEGMENTS, size=N_USERS, p=[0.40, 0.30, 0.20, 0.10])
risk_profile = rng.choice(RISK, size=N_USERS, p=[0.25, 0.50, 0.25])

# whales ~ 1.5%
whale_flag = (rng.random(N_USERS) < 0.015).astype(int)

# deposits (skewed)
initial_deposit_usd = np.where(
    whale_flag == 1,
    rng.lognormal(mean=7.2, sigma=0.6, size=N_USERS),   # whales
    rng.lognormal(mean=4.2, sigma=0.9, size=N_USERS)    # others
)
initial_deposit_usd = np.round(initial_deposit_usd, 2)

# referral links (only some)
referrer_user_id = np.full(N_USERS, np.nan)
is_ref = (acq_channel == "referral") & (rng.random(N_USERS) < 0.8)
referrer_user_id[is_ref] = rng.integers(1, N_USERS + 1, size=is_ref.sum())

is_kyc = (rng.random(N_USERS) < 0.35).astype(int)

campaign_id = np.array([f"cmp_{c}_{rng.integers(1, 35)}" for c in acq_channel])
ad_group = np.array([f"adg_{rng.integers(1, 120)}" if c == "ads" else "" for c in acq_channel])

users = pd.DataFrame({
    "user_id": user_ids,
    "created_at": created_at,
    "country": country,
    "timezone": timezone,
    "acq_channel": acq_channel,
    "campaign_id": campaign_id,
    "ad_group": ad_group,
    "device_os": device_os,
    "client": client,
    "app_version": app_version,
    "wallet_type": wallet_type,
    "chain_pref": chain_pref,
    "referrer_user_id": referrer_user_id,
    "is_kyc": is_kyc,
    "player_segment": player_segment,
    "risk_profile": risk_profile,
    "whale_flag": whale_flag,
    "initial_deposit_usd": initial_deposit_usd
})

users.to_csv(os.path.join(OUT_DIR, "users.csv"), index=False)
prog(f"users saved: {len(users):,} rows")

# ----------------------------
# PLANETS
# ----------------------------
prog("Generating planets...")

base_planets = rng.poisson(lam=np.where(player_segment == "builder", 2.2, 1.2), size=N_USERS)
base_planets += whale_flag * rng.integers(1, 4, size=N_USERS)
base_planets = np.clip(base_planets, 0, 8)

planet_rows = []
planet_id = 1

planet_classes = ["ice", "desert", "forest", "volcanic", "ocean"]
specs = ["mining", "energy", "food", "research", "trade_hub"]

for i, (uid, npl, u_created) in enumerate(zip(user_ids, base_planets, created_at), start=1):
    for _ in range(int(npl)):
        created = u_created + pd.to_timedelta(rng.integers(0, 20), unit="D") + pd.to_timedelta(rng.integers(0, 86400), unit="s")
        created = min(created, end_dt - pd.Timedelta(seconds=1))
        planet_rows.append({
            "planet_id": planet_id,
            "user_id": int(uid),
            "created_at": created,
            "planet_class": rng.choice(planet_classes),
            "sector_id": int(rng.integers(1, 51)),
            "specialization": rng.choice(specs, p=[0.25, 0.20, 0.20, 0.20, 0.15]),
            "pdi_base": float(np.round(rng.normal(50, 12), 2))
        })
        planet_id += 1

    if i % 2000 == 0:
        prog_every(i, N_USERS, 2000, "planets users processed")

planets = pd.DataFrame(planet_rows)
planets.to_csv(os.path.join(OUT_DIR, "planets.csv"), index=False)
prog(f"planets saved: {len(planets):,} rows")

# ----------------------------
# ACTIVE DAYS / CHURN SIMULATION
# ----------------------------
prog("Simulating lifetimes (churn model)...")

channel_factor = pd.Series(acq_channel).map({"organic": 1.1, "referral": 1.2, "ads": 0.9, "influencer": 1.0, "partner": 1.0}).to_numpy()
segment_factor = pd.Series(player_segment).map({"builder": 1.1, "trader": 1.0, "investor": 0.95, "social": 1.05}).to_numpy()
whale_factor = np.where(whale_flag == 1, 1.6, 1.0)

mean_life = 18 * channel_factor * segment_factor * whale_factor
life_days = rng.geometric(p=np.clip(1/mean_life, 0.01, 0.3), size=N_USERS)
life_days = np.clip(life_days, 1, DAYS)

# ----------------------------
# SESSIONS
# ----------------------------
prog("Generating sessions...")

session_rows = []
session_id = 1

for idx, (uid, u_created, ld, seg, wh) in enumerate(zip(user_ids, created_at, life_days, player_segment, whale_flag), start=1):
    start_day = pd.Timestamp(u_created).floor("D")
    last_day = min(start_day + pd.Timedelta(days=int(ld)), days[-1])
    active_range = pd.date_range(start_day, last_day, freq="D")

    lam = 1.2
    if seg == "trader":  lam = 1.6
    if seg == "builder": lam = 1.4
    if wh == 1:          lam *= 1.8

    for d in active_range:
        t = (d - start_day).days
        decay = np.exp(-t / (35 if wh == 1 else 22))
        k = rng.poisson(lam=max(0.05, lam * decay))
        if k == 0:
            continue

        for _ in range(int(k)):
            start = d + pd.to_timedelta(int(rng.integers(0, 86400)), unit="s")
            length = int(np.clip(rng.lognormal(mean=7.2 if seg != "social" else 6.9, sigma=0.6), 60, 7200))
            end = start + pd.to_timedelta(length, unit="s")
            entry = rng.choice(["push", "direct", "market", "deeplink"], p=[0.15, 0.55, 0.20, 0.10])

            session_rows.append({
                "session_id": session_id,
                "user_id": int(uid),
                "start_ts": start,
                "end_ts": end,
                "session_len_sec": length,
                "entry_point": entry,
                "net_latency_ms": int(np.clip(rng.normal(120, 60), 20, 600)),
                "crash_flag": int(rng.random() < 0.01)
            })
            session_id += 1

    if idx % 1000 == 0:
        prog_every(idx, N_USERS, 1000, "sessions users processed")

sessions = pd.DataFrame(session_rows)
sessions.to_csv(os.path.join(OUT_DIR, "sessions.csv"), index=False)
prog(f"sessions saved: {len(sessions):,} rows")

# ----------------------------
# ECONOMIC EVENTS (SHOCKS)
# ----------------------------
prog("Generating economic events...")

econ_rows = []
event_types = ["solar_storm", "supply_crunch", "oracle_glitch", "transport_strike"]

for i in range(1, 13):
    s = start_dt + pd.Timedelta(days=int(rng.integers(0, max(1, DAYS-3))))
    e = s + pd.Timedelta(days=int(rng.integers(1, 4)))
    res = rng.choice(RESOURCES)
    econ_rows.append({
        "econ_event_id": i,
        "start_ts": s,
        "end_ts": min(e, end_dt),
        "event_type": rng.choice(event_types),
        "affected_resource": res,
        "supply_multiplier": float(np.round(rng.uniform(0.6, 1.2), 3)),
        "demand_multiplier": float(np.round(rng.uniform(0.8, 1.4), 3)),
        "notes": "synthetic shock"
    })

economic_events = pd.DataFrame(econ_rows)
economic_events["start_ts"] = pd.to_datetime(economic_events["start_ts"])
economic_events["end_ts"] = pd.to_datetime(economic_events["end_ts"])
economic_events.to_csv(os.path.join(OUT_DIR, "economic_events.csv"), index=False)
prog(f"economic_events saved: {len(economic_events):,} rows")

# ----------------------------
# PRICE ORACLE DAILY
# ----------------------------
prog("Generating price_oracle_daily...")

price_rows = []
prices = {r: float(rng.uniform(1.0, 5.0)) for r in RESOURCES}

def apply_shock(day_ts: pd.Timestamp, r: str, base_price: float) -> float:
    day_d = day_ts.date()
    shocks = economic_events[
        (economic_events["affected_resource"] == r) &
        (economic_events["start_ts"].dt.date <= day_d) &
        (economic_events["end_ts"].dt.date >= day_d)
    ]
    if shocks.empty:
        return base_price

    dm = shocks["demand_multiplier"].mean()
    sm = shocks["supply_multiplier"].mean()
    factor = (dm / sm)
    return base_price * float(np.clip(factor, 0.6, 1.8))

for i, d in enumerate(days, start=1):
    for r in RESOURCES:
        drift = rng.normal(0.0, 0.03)
        prices[r] = max(0.1, prices[r] * (1.0 + drift))
        p = apply_shock(pd.Timestamp(d), r, prices[r])
        vol = float(np.round(abs(rng.normal(0.0, 0.12)), 4))
        price_rows.append({"day": d.date(), "resource": r, "price_token": float(np.round(p, 4)), "volatility": vol})

    if i % 10 == 0:
        prog_every(i, len(days), 10, "price oracle days")

price_oracle = pd.DataFrame(price_rows)
price_oracle.to_csv(os.path.join(OUT_DIR, "price_oracle_daily.csv"), index=False)
prog(f"price_oracle_daily saved: {len(price_oracle):,} rows")

# ----------------------------
# RESOURCE DAILY (HEAVY)
# ----------------------------
prog("Generating resource_daily (this is heavy)...")

resource_rows = []
n_planets = len(planets)

for i, row in enumerate(planets.itertuples(index=False), start=1):
    p_created = pd.to_datetime(row.created_at).floor("D")
    active_days = pd.date_range(max(p_created, start_dt), days[-1], freq="D")

    spec = row.specialization
    spec_boost = {r: 1.0 for r in RESOURCES}
    if spec == "mining":   spec_boost["ORE"] = 1.4
    if spec == "energy":   spec_boost["ENERGY"] = 1.4
    if spec == "food":     spec_boost["FOOD"] = 1.4
    if spec == "research": spec_boost["CRYSTAL"] = 1.3

    for d in active_days:
        t = (d - p_created).days
        capacity = 100 + 4*t + float(rng.normal(0, 8))
        efficiency = float(np.clip(rng.normal(0.78, 0.10), 0.35, 0.98))
        build_q = int(np.clip(rng.poisson(1.2), 0, 8))

        for r in RESOURCES:
            base = (20 + 0.6*t) * spec_boost[r] * efficiency
            produced = max(0.0, rng.normal(base, base*0.25))
            consumed = max(0.0, rng.normal(produced*0.55, produced*0.15))
            stored_end = max(0.0, produced - consumed + rng.normal(0, 3))

            resource_rows.append({
                "day": d.date(),
                "planet_id": int(row.planet_id),
                "user_id": int(row.user_id),
                "resource": r,
                "produced": float(np.round(produced, 3)),
                "consumed": float(np.round(consumed, 3)),
                "stored_end": float(np.round(min(stored_end, capacity), 3)),
                "capacity": float(np.round(capacity, 2)),
                "efficiency": float(np.round(efficiency, 4)),
                "build_queue_len": build_q
            })

    if i % 250 == 0:
        prog_every(i, n_planets, 250, "resource_daily planets processed")

resource_daily = pd.DataFrame(resource_rows)
resource_daily.to_csv(os.path.join(OUT_DIR, "resource_daily.csv"), index=False)
prog(f"resource_daily saved: {len(resource_daily):,} rows")

# ----------------------------
# MARKET ORDERS / TRADES
# ----------------------------
prog("Generating market_orders / market_trades...")

# derive active users/day from sessions
active_users = sessions[["user_id", "start_ts"]].copy()
active_users["start_ts"] = pd.to_datetime(active_users["start_ts"])
active_users["day"] = active_users["start_ts"].dt.date
au_day = active_users.groupby("day")["user_id"].unique()

orders = []
trades = []
order_id = 1
trade_id = 1

def volume_bucket(v: float) -> str:
    if v < 200: return "low"
    if v < 2000: return "mid"
    if v < 10000: return "high"
    return "whale"

trade_intensity = np.where(whale_flag == 1, rng.uniform(2.5, 6.0, N_USERS), rng.uniform(0.3, 2.0, N_USERS))
trade_intensity *= np.where(player_segment == "trader", 1.7, 1.0)

tax_map = {"low": 0.05, "mid": 0.06, "high": 0.075, "whale": 0.10}

for di, d in enumerate(days, start=1):
    day_users = au_day.get(d.date(), np.array([], dtype=int))
    if len(day_users) == 0:
        if di % 5 == 0:
            prog_every(di, len(days), 5, "market days processed")
        continue

    m = int(len(day_users) * 0.22)
    if m <= 0:
        if di % 5 == 0:
            prog_every(di, len(days), 5, "market days processed")
        continue

    trade_users = rng.choice(day_users, size=max(1, m), replace=False)

    for uid in trade_users:
        uidx = int(uid) - 1
        k = rng.poisson(lam=trade_intensity[uidx])
        if k == 0:
            continue

        approx_7d_vol = float(initial_deposit_usd[uidx] * rng.uniform(0.02, 0.15) * trade_intensity[uidx])
        b = volume_bucket(approx_7d_vol)
        tax_rate = tax_map[b]

        for _ in range(int(k)):
            ts = pd.Timestamp(d) + pd.to_timedelta(int(rng.integers(0, 86400)), unit="s")
            res = rng.choice(RESOURCES)
            side = rng.choice(["buy", "sell"], p=[0.48, 0.52])

            px = float(price_oracle[(price_oracle["day"] == d.date()) & (price_oracle["resource"] == res)]["price_token"].iloc[0])
            amount = float(np.round(rng.lognormal(mean=2.0, sigma=0.7), 4))
            if whale_flag[uidx] == 1:
                amount *= float(rng.uniform(2.5, 8.0))

            vol = float(price_oracle[(price_oracle["day"] == d.date()) & (price_oracle["resource"] == res)]["volatility"].iloc[0])
            spread_bps = float(np.round(np.clip(vol * 1500, 5, 120), 2))
            slippage_bps = float(np.round(np.clip(vol * 1800 * rng.uniform(0.6, 1.4), 2, 220), 2))

            gross = amount * px
            fee = gross * 0.002
            tax = gross * tax_rate

            orders.append({
                "order_id": order_id,
                "ts_created": ts,
                "ts_closed": ts + pd.to_timedelta(int(rng.integers(5, 1200)), unit="s"),
                "user_id": int(uid),
                "side": side,
                "asset_base": res,
                "asset_quote": "TOKEN",
                "amount_base": amount,
                "price": px,
                "order_type": rng.choice(["limit", "market"], p=[0.72, 0.28]),
                "status": "filled",
                "tax_rate": float(np.round(tax_rate, 4)),
                "expected_slippage_bps": slippage_bps
            })

            maker = int(uid)
            taker = int(rng.choice(day_users))
            if taker == maker:
                taker = int(rng.integers(1, N_USERS + 1))

            bull_bear = rng.choice(["bull", "bear", "sideways"], p=[0.35, 0.30, 0.35])

            trades.append({
                "trade_id": trade_id,
                "ts": ts,
                "order_id": order_id,
                "maker_user_id": maker,
                "taker_user_id": taker,
                "asset_base": res,
                "asset_quote": "TOKEN",
                "amount_base": amount,
                "price": px,
                "fee_amount": float(np.round(fee, 6)),
                "tax_amount": float(np.round(tax, 6)),
                "slippage_bps": slippage_bps,
                "spread_bps": spread_bps,
                "bull_bear_regime": bull_bear,
                "user_7d_volume_bucket": b
            })

            order_id += 1
            trade_id += 1

    if di % 5 == 0:
        prog_every(di, len(days), 5, "market days processed")

market_orders = pd.DataFrame(orders)
market_trades = pd.DataFrame(trades)

market_orders.to_csv(os.path.join(OUT_DIR, "market_orders.csv"), index=False)
market_trades.to_csv(os.path.join(OUT_DIR, "market_trades.csv"), index=False)

prog(f"market_orders saved: {len(market_orders):,} rows")
prog(f"market_trades saved: {len(market_trades):,} rows")

# ----------------------------
# TOKEN LEDGER
# ----------------------------
prog("Generating token_ledger: rewards from production...")

ledger_rows = []
ledger_id = 1

prod_daily = resource_daily.groupby(["day", "user_id"])["produced"].sum().reset_index()
n_rewards = len(prod_daily)

for j, r in enumerate(prod_daily.itertuples(index=False), start=1):
    amt = float(np.round(max(0.0, r.produced * rng.uniform(0.02, 0.06)), 6))
    ts = pd.Timestamp(r.day) + pd.to_timedelta(int(rng.integers(0, 86400)), unit="s")
    ledger_rows.append({
        "ledger_id": ledger_id,
        "ts": ts,
        "user_id": int(r.user_id),
        "tx_type": "reward",
        "token_symbol": "TOKEN",
        "amount": amt,
        "chain": rng.choice(CHAINS),
        "gas_fee_token": float(np.round(rng.uniform(0.0001, 0.01), 6)),
        "tx_hash": make_tx_hash(rng)
    })
    ledger_id += 1

    if j % 200000 == 0:
        prog_every(j, n_rewards, 200000, "token_ledger rewards")

prog("Generating token_ledger: taxes & fees from trades...")

n_trades = len(market_trades)
for k, t in enumerate(market_trades.itertuples(index=False), start=1):
    # tax
    ledger_rows.append({
        "ledger_id": ledger_id,
        "ts": t.ts,
        "user_id": int(t.maker_user_id),
        "tx_type": "tax",
        "token_symbol": "TOKEN",
        "amount": -float(np.round(t.tax_amount, 6)),
        "chain": rng.choice(CHAINS),
        "gas_fee_token": float(np.round(rng.uniform(0.0, 0.006), 6)),
        "tx_hash": make_tx_hash(rng)
    })
    ledger_id += 1

    # fee
    ledger_rows.append({
        "ledger_id": ledger_id,
        "ts": t.ts,
        "user_id": int(t.maker_user_id),
        "tx_type": "market_fee",
        "token_symbol": "TOKEN",
        "amount": -float(np.round(t.fee_amount, 6)),
        "chain": rng.choice(CHAINS),
        "gas_fee_token": float(np.round(rng.uniform(0.0, 0.006), 6)),
        "tx_hash": make_tx_hash(rng)
    })
    ledger_id += 1

    if k % 200000 == 0:
        prog_every(k, n_trades, 200000, "token_ledger trades processed")

prog("Generating token_ledger: staking/unstaking...")

stake_users = rng.choice(user_ids, size=int(N_USERS * 0.18), replace=False)
n_stake_users = len(stake_users)

for i, uid in enumerate(stake_users, start=1):
    n = int(rng.integers(1, 6))
    for _ in range(n):
        ts = start_dt + pd.to_timedelta(int(rng.integers(0, DAYS*86400)), unit="s")
        amt = float(np.round(rng.lognormal(mean=2.2, sigma=0.9), 6))
        tx_type = rng.choice(["stake", "unstake"], p=[0.6, 0.4])
        sign = 1 if tx_type == "stake" else -1
        ledger_rows.append({
            "ledger_id": ledger_id,
            "ts": ts,
            "user_id": int(uid),
            "tx_type": tx_type,
            "token_symbol": "TOKEN",
            "amount": float(np.round(amt * sign, 6)),
            "chain": rng.choice(CHAINS),
            "gas_fee_token": float(np.round(rng.uniform(0.0001, 0.02), 6)),
            "tx_hash": make_tx_hash(rng)
        })
        ledger_id += 1

    if i % 1000 == 0:
        prog_every(i, n_stake_users, 1000, "staking users processed")

token_ledger = pd.DataFrame(ledger_rows)
token_ledger.to_csv(os.path.join(OUT_DIR, "token_ledger.csv"), index=False)
prog(f"token_ledger saved: {len(token_ledger):,} rows")

# ----------------------------
# ALLIANCES + MEMBERSHIP
# ----------------------------
prog("Generating alliances & membership...")

n_alliances = 120
alliances = pd.DataFrame({
    "alliance_id": np.arange(1, n_alliances + 1),
    "created_ts": start_dt + pd.to_timedelta(rng.integers(0, DAYS*86400, n_alliances), unit="s"),
    "name": [f"Alliance_{i:03d}" for i in range(1, n_alliances + 1)],
    "policy_type": rng.choice(["free_trade", "cartel", "balanced"], n_alliances, p=[0.55, 0.15, 0.30]),
    "tax_rebate_pct": np.round(rng.uniform(0.00, 0.05, n_alliances), 4)
})
alliances.to_csv(os.path.join(OUT_DIR, "alliances.csv"), index=False)

mem_rows = []
for i, uid in enumerate(user_ids, start=1):
    if rng.random() < 0.42:
        aid = int(rng.integers(1, n_alliances + 1))
        join = pd.Timestamp(users.loc[users.user_id == uid, "created_at"].iloc[0]) + pd.to_timedelta(int(rng.integers(1, 20)), unit="D")
        leave = join + pd.to_timedelta(int(rng.integers(10, 90)), unit="D") if rng.random() < 0.18 else pd.NaT
        mem_rows.append({
            "alliance_id": aid,
            "user_id": int(uid),
            "join_ts": join,
            "leave_ts": leave,
            "role": rng.choice(["member", "treasurer", "analyst"], p=[0.88, 0.07, 0.05])
        })

    if i % 5000 == 0:
        prog_every(i, N_USERS, 5000, "membership users processed")

membership = pd.DataFrame(mem_rows)
membership.to_csv(os.path.join(OUT_DIR, "alliance_membership.csv"), index=False)

prog(f"alliances saved: {len(alliances):,} rows")
prog(f"membership saved: {len(membership):,} rows")

# ----------------------------
# CONTRACTS (FIXED sday + progress)
# ----------------------------
prog("Generating contracts...")

contract_rows = []
contract_id = 1
n_contracts_target = 4500

for k in range(1, n_contracts_target + 1):
    supplier = int(rng.integers(1, N_USERS + 1))
    buyer = int(rng.integers(1, N_USERS + 1))
    if buyer == supplier:
        continue

    res = rng.choice(RESOURCES)

    # FIX: rng.choice(days) returns numpy.datetime64 (no .date), so pick by index
    sday = days[int(rng.integers(0, len(days)))].date()
    eday = (pd.Timestamp(sday) + pd.Timedelta(days=int(rng.integers(7, 28)))).date()
    eday = min(eday, days[-1].date())

    contract_rows.append({
        "contract_id": contract_id,
        "supplier_user_id": supplier,
        "buyer_user_id": buyer,
        "resource": res,
        "daily_qty": float(np.round(rng.uniform(5, 120), 3)),
        "price_formula": rng.choice(["fixed", "indexed_oracle", "oracle_plus_spread"], p=[0.35, 0.45, 0.20]),
        "start_day": sday,
        "end_day": eday,
        "penalty_pct": float(np.round(rng.uniform(0.01, 0.15), 4)),
        "fulfilled_ratio": float(np.round(np.clip(rng.normal(0.86, 0.18), 0.0, 1.0), 4))
    })
    contract_id += 1

    if k % 500 == 0:
        prog_every(k, n_contracts_target, 500, "contracts")

contracts = pd.DataFrame(contract_rows)
contracts.to_csv(os.path.join(OUT_DIR, "contracts.csv"), index=False)
prog(f"contracts saved: {len(contracts):,} rows")

# ----------------------------
# GOVERNANCE (PROPOSALS + VOTES) + progress
# ----------------------------
prog("Generating governance proposals & votes...")

proposal_rows = []
vote_rows = []
proposal_id = 1
vote_id = 1

topics = ["tax_change", "subsidy", "emission", "burn", "market_rules"]
scopes = ["global", "sector", "alliance"]

n_props = 180
for p in range(n_props):
    created = start_dt + pd.to_timedelta(int(rng.integers(0, DAYS*86400)), unit="s")
    topic = rng.choice(topics)
    scope = rng.choice(scopes, p=[0.55, 0.25, 0.20])
    param_target = rng.choice(RESOURCES + ["volume_bucket", "new_users", "all_users"])
    param_value = float(np.round(rng.uniform(0.01, 0.12), 4))
    vs = created + pd.Timedelta(days=1)
    ve = vs + pd.Timedelta(days=int(rng.integers(2, 6)))
    status = rng.choice(["passed", "rejected", "expired"], p=[0.45, 0.45, 0.10])

    proposal_rows.append({
        "proposal_id": proposal_id,
        "created_ts": created,
        "scope": scope,
        "topic": topic,
        "param_target": param_target,
        "param_value": param_value,
        "vote_start": vs,
        "vote_end": ve,
        "status": status
    })

    voters = rng.choice(user_ids, size=int(rng.integers(50, 600)), replace=False)
    for u in voters:
        vp = float(np.round(rng.lognormal(mean=1.8, sigma=1.0), 4))
        locked = float(np.round(vp * rng.uniform(0.2, 2.0), 4))
        vote_rows.append({
            "vote_id": vote_id,
            "proposal_id": proposal_id,
            "user_id": int(u),
            "ts": vs + pd.to_timedelta(int(rng.integers(0, int((ve-vs).total_seconds()))), unit="s"),
            "vote": rng.choice(["yes", "no", "abstain"], p=[0.48, 0.42, 0.10]),
            "voting_power": vp,
            "tokens_locked": locked
        })
        vote_id += 1

    proposal_id += 1

    if (p + 1) % 20 == 0:
        prog_every(p + 1, n_props, 20, f"governance proposals (votes so far: {len(vote_rows):,})")

governance_proposals = pd.DataFrame(proposal_rows)
governance_votes = pd.DataFrame(vote_rows)

governance_proposals.to_csv(os.path.join(OUT_DIR, "governance_proposals.csv"), index=False)
governance_votes.to_csv(os.path.join(OUT_DIR, "governance_votes.csv"), index=False)

prog(f"governance_proposals saved: {len(governance_proposals):,} rows")
prog(f"governance_votes saved: {len(governance_votes):,} rows")

# ----------------------------
# EXPERIMENTS + ASSIGNMENTS
# ----------------------------
prog("Generating experiments & assignments...")

experiments = pd.DataFrame([
    {
        "experiment_id": "EXP_TAX_DYNAMIC",
        "name": "Dynamic market tax by 7d volume bucket",
        "start_ts": start_dt + pd.Timedelta(days=max(1, DAYS//4)),
        "end_ts": start_dt + pd.Timedelta(days=min(DAYS-1, (DAYS//4)*3)),
        "primary_metric": "DWD + MSI",
        "variant_params_json": json.dumps({
            "A": {"tax_by_bucket": {"low": 0.05, "mid": 0.05, "high": 0.05, "whale": 0.05}},
            "B": {"tax_by_bucket": {"low": 0.05, "mid": 0.06, "high": 0.075, "whale": 0.10}}
        })
    },
    {
        "experiment_id": "EXP_NFT_RENT",
        "name": "NFT rent marketplace",
        "start_ts": start_dt + pd.Timedelta(days=max(1, DAYS//9)),
        "end_ts": start_dt + pd.Timedelta(days=min(DAYS-1, (DAYS//9)*5)),
        "primary_metric": "ENR + RUR",
        "variant_params_json": json.dumps({
            "A": {"rent_enabled": False},
            "B": {"rent_enabled": True, "min_rent_days": 3}
        })
    },
])

assign_rows = []
for exp in experiments.itertuples(index=False):
    eligible = users[pd.to_datetime(users["created_at"]) < pd.to_datetime(exp.end_ts)]["user_id"].to_numpy()
    if len(eligible) == 0:
        continue
    sample = rng.choice(eligible, size=max(1, int(len(eligible) * 0.35)), replace=False)
    for uid in sample:
        assign_rows.append({
            "experiment_id": exp.experiment_id,
            "user_id": int(uid),
            "variant": rng.choice(["A", "B"]),
            "assigned_ts": pd.to_datetime(exp.start_ts) + pd.to_timedelta(int(rng.integers(0, 5*86400)), unit="s")
        })

experiment_assignments = pd.DataFrame(assign_rows)

experiments.to_csv(os.path.join(OUT_DIR, "experiments.csv"), index=False)
experiment_assignments.to_csv(os.path.join(OUT_DIR, "experiment_assignments.csv"), index=False)

prog(f"experiments saved: {len(experiments):,} rows")
prog(f"experiment_assignments saved: {len(experiment_assignments):,} rows")

# ----------------------------
# MARKETING COSTS
# ----------------------------
prog("Generating marketing_costs...")

mc_rows = []
for i, d in enumerate(days, start=1):
    for ch in CHANNELS:
        spend = float(np.round(max(0, rng.normal(1800 if ch == "ads" else 500, 300)), 2))
        impr = int(max(0, rng.normal(250000 if ch == "ads" else 70000, 25000)))
        clicks = int(impr * rng.uniform(0.005, 0.02))
        installs = int(clicks * rng.uniform(0.08, 0.22))
        mc_rows.append({
            "day": d.date(),
            "acq_channel": ch,
            "campaign_id": f"cmp_{ch}_{int(rng.integers(1, 35))}",
            "spend_usd": spend,
            "impressions": impr,
            "clicks": clicks,
            "installs": installs
        })

    if i % 10 == 0:
        prog_every(i, len(days), 10, "marketing days")

marketing_costs = pd.DataFrame(mc_rows)
marketing_costs.to_csv(os.path.join(OUT_DIR, "marketing_costs.csv"), index=False)
prog(f"marketing_costs saved: {len(marketing_costs):,} rows")

# ----------------------------
# DONE
# ----------------------------
prog("ALL DONE ✅")
prog("Generated files:")
for f in sorted(os.listdir(OUT_DIR)):
    prog(f" - {f}")


[    0.0s] Start. OUT_DIR=synthetic_defi_game_data, N_USERS=20000, DAYS=90
[    0.0s] Generating users...
[    0.7s] users saved: 20,000 rows
[    0.7s] Generating planets...
[    1.8s] planets users processed: 2000/20000 (10.0%)
[    2.8s] planets users processed: 4000/20000 (20.0%)
[    3.9s] planets users processed: 6000/20000 (30.0%)
[    4.9s] planets users processed: 8000/20000 (40.0%)
[    6.0s] planets users processed: 10000/20000 (50.0%)
[    7.1s] planets users processed: 12000/20000 (60.0%)
[    8.1s] planets users processed: 14000/20000 (70.0%)
[    9.2s] planets users processed: 16000/20000 (80.0%)
[   10.2s] planets users processed: 18000/20000 (90.0%)
[   11.2s] planets users processed: 20000/20000 (100.0%)
[   11.6s] planets saved: 32,338 rows
[   11.6s] Simulating lifetimes (churn model)...
[   11.7s] Generating sessions...
[   16.0s] sessions users processed: 1000/20000 (5.0%)
[   20.4s] sessions users processed: 2000/20000 (10.0%)
[   24.7s] sessions users processed: